# 🔬 XGBoost/LightGBM sur les Embeddings SSL de PatchTST
## Test demandé par Dr. Sarun (email) — valider la qualité des représentations SSL
### CMKL University · Stage 2026

---

**Question de Dr. Sarun** :
> *"Train the PatchTST using the self-supervised in the first step, then use
> the results to train XGBoost and LightGBM. [...] It should produce results
> as good as or better than models trained on the spectra data directly, if
> the embeddings capture the key features of the data."*

**Point méthodologique important** : on utilise le backbone **juste après la
Phase 1 SSL** (`ssl_backbone_mixed.pth`), AVANT toute Phase 2/3 supervisée.
C'est la vraie question posée — les représentations apprises **sans aucun
label** sont-elles déjà exploitables ?

**Ce notebook est rapide** — aucun entraînement PatchTST nécessaire, juste :
```
1. Charger le backbone SSL déjà entraîné (Phase 1 uniquement)
2. Passer chaque spectre dedans → extraire l'embedding (mean pooling)
3. Entraîner XGBoost/LightGBM sur ces embeddings
4. Comparer à :
   - XGB/LGB sur spectres bruts/PCA (déjà obtenu : ~93-94%)
   - PatchTST classification complète (déjà obtenu : ~94-95%)
```

**Pourquoi mean pooling et pas Attention Pooling** : l'Attention Pooling est
un composant **appris** qui n'existe que dans `ClassificationHead`, entraîné
en Phase 2/3. Au stade Phase 1 seul, ces poids n'existent pas encore (ou
seraient aléatoires) — le mean pooling est le choix standard et interprétable
pour extraire un vecteur unique depuis un backbone SSL pur.


---
## ⚙️ Section 0 — Imports & Configuration


In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'xgboost', 'lightgbm', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

In [ ]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')

In [ ]:
# ── Config IDENTIQUE au modèle mixte (nécessaire pour charger les poids) ──
CFG = {
    'L'          : 6700,
    'patch_size' : 320,
    'stride'     : 256,
    'd_model'    : 256,
    'n_heads'    : 16,
    'n_layers'   : 5,
    'd_ff'       : 512,
    'dropout'    : 0.1,
    'N_CLASSES'  : 30,
}
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
WN_GRID = np.arange(650, 4000, 0.5)
print(f'N_PATCHES = {N_PATCHES}')

ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

---
## 🏛️ Section 1 — Architecture (PatchEmbedding + Conformer Backbone SEULEMENT)

Pas besoin des têtes de classification/denoising — on veut juste le backbone.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

print('✓ Architecture (backbone) définie')

In [ ]:
patch_embed = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone    = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                   CFG['d_ff'], CFG['dropout']).to(DEVICE)

# ── Charger le backbone SSL (Phase 1 UNIQUEMENT — pas de Phase 2/3) ────────
SSL_PATH = os.path.join(HOME, 'models', 'ssl_backbone_mixed.pth')
ckpt = torch.load(SSL_PATH, map_location=DEVICE, weights_only=False)
patch_embed.load_state_dict(ckpt['patch_embed'])
backbone.load_state_dict(ckpt['backbone'])
patch_embed.eval(); backbone.eval()

print(f'✓ Backbone SSL chargé (Phase 1 uniquement, PAS de fine-tuning supervisé)')
print(f'  Val MSE au moment de la sauvegarde : {ckpt["ssl_val_loss"]:.6f}')

In [ ]:
@torch.no_grad()
def extract_embeddings(spectra_array, batch_size=64):
    """
    Passe des spectres bruts dans le backbone SSL et retourne un embedding
    par spectre (mean pooling sur les patches).
    """
    embeddings = []
    n = len(spectra_array)
    for i in range(0, n, batch_size):
        batch = spectra_array[i:i+batch_size]
        x = torch.tensor(batch, dtype=torch.float32)
        mu, sigma = x.mean(dim=1, keepdim=True), x.std(dim=1, keepdim=True) + 1e-8
        x = ((x - mu) / sigma).to(DEVICE)

        tokens = patch_embed(x)
        z = backbone(tokens)          # (B, N, D)
        z_pool = z.mean(dim=1)        # mean pooling → (B, D)
        embeddings.append(z_pool.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

print('✓ Fonction d\'extraction définie')
print(f'  Dimension de sortie par embedding : {CFG["d_model"]}')

---
## 📊 Section 2 — Chargement des données (identique au sweep de sensibilité)


In [ ]:
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed')
TRAIN_DIR = os.path.join(NPY_ROOT, '1.1 TrainingSet - UptoY dB')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p): print(f'  ✗ INTROUVABLE : {p}')
    return p

noise = 'Upto30SNR'
npy_train_noisy = np.load(npy_path(TRAIN_DIR, f'TrainNoisySet_{noise}_Pre.npy'))
npy_test_noisy  = np.load(npy_path(TEST_DIR,  f'TestNoisySet_{noise}_Pre.npy'))
npy_train_clean_shape = np.load(npy_path(TRAIN_DIR, 'TrainGroundTruthSet_Pre.npy')).shape

N_PER_CLASS_NPY_TRAIN = npy_train_clean_shape[0] // N_CLASSES
N_PER_CLASS_NPY_TEST  = npy_test_noisy.shape[0]  // N_CLASSES
npy_labels_train_full = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TRAIN)
npy_labels_test       = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TEST)

# ── Même val reservation que PatchTST pour un train EFFECTIF identique ────
N_VAL_PER_CLASS_NPY = 30
train_idx_npy = []
for c in range(N_CLASSES):
    cls_idx = np.where(npy_labels_train_full == c)[0]
    rng = np.random.RandomState(SEED)
    rng.shuffle(cls_idx)
    train_idx_npy.extend(cls_idx[N_VAL_PER_CLASS_NPY:])
train_idx_npy = np.array(train_idx_npy)

npy_train_noisy_eff  = npy_train_noisy[train_idx_npy]
npy_labels_train_eff = npy_labels_train_full[train_idx_npy]

print(f'✓ .npy Train effectif : {len(npy_train_noisy_eff)}')
print(f'✓ .npy Test officiel  : {len(npy_test_noisy)}')

In [ ]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')
PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}

EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

In [ ]:
print('Chargement des CSV bruités...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        if not is_noisy_csv(fp): continue
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'spectrum': sp_interp})

df_csv_noisy = pd.DataFrame(csv_records)
df_csv_noisy['label_enc'] = le.transform(df_csv_noisy['label'])
print(f'Total CSV bruités : {len(df_csv_noisy)}')

# ── Split IDENTIQUE au sweep de sensibilité (fraction=1.0, référence) ─────
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]
df_csv_single = df_csv_noisy[ df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3, random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_train = pd.concat([df_csv_multi.iloc[idx_tr_csv], df_csv_single]).reset_index(drop=True)
df_csv_test  = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'CSV Train : {len(df_csv_train)}   CSV Test (FIXE) : {len(df_csv_test)}')

---
## 🧠 Section 3 — Extraction des embeddings SSL (rapide, pas d'entraînement)


In [ ]:
csv_train_spectra = np.stack(df_csv_train['spectrum'].values)
csv_test_spectra  = np.stack(df_csv_test['spectrum'].values)

print('Extraction des embeddings...')
emb_npy_train = extract_embeddings(npy_train_noisy_eff)
print(f'  ✓ .npy train : {emb_npy_train.shape}')

emb_csv_train = extract_embeddings(csv_train_spectra)
print(f'  ✓ CSV train  : {emb_csv_train.shape}')

emb_npy_test = extract_embeddings(npy_test_noisy)
print(f'  ✓ .npy test  : {emb_npy_test.shape}')

emb_csv_test = extract_embeddings(csv_test_spectra)
print(f'  ✓ CSV test   : {emb_csv_test.shape}')

# ── Train combiné (comme pour le sweep RF/GB "mixte") ─────────────────────
X_train_emb = np.concatenate([emb_npy_train, emb_csv_train], axis=0)
y_train = np.concatenate([npy_labels_train_eff, df_csv_train['label_enc'].values])

print(f'\n✓ X_train_emb (combiné) : {X_train_emb.shape}')
print(f'  Dimension embedding : {CFG["d_model"]} (vs 6700 pour un spectre brut, ou 50 pour PCA)')

---
## ⚡ Section 4 — XGBoost & LightGBM sur les embeddings


In [ ]:
XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
                   eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
LGB_PARAMS = dict(n_estimators=300, num_leaves=63, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
                   class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)

results = {}
for model_name, ModelClass, params in [
    ('XGBoost',  xgb.XGBClassifier, XGB_PARAMS),
    ('LightGBM', lgb.LGBMClassifier, LGB_PARAMS),
]:
    print(f'=== {model_name} sur embeddings SSL ===')
    clf = ModelClass(**params)
    clf.fit(X_train_emb, y_train)

    preds_npy = clf.predict(emb_npy_test)
    preds_csv = clf.predict(emb_csv_test)
    acc_npy = accuracy_score(npy_labels_test, preds_npy)
    acc_csv = accuracy_score(df_csv_test['label_enc'].values, preds_csv)
    f1_npy = f1_score(npy_labels_test, preds_npy, average='macro', zero_division=0)
    f1_csv = f1_score(df_csv_test['label_enc'].values, preds_csv, average='macro', zero_division=0)

    results[model_name] = {'model': clf, 'acc_npy': acc_npy, 'acc_csv': acc_csv,
                           'f1_npy': f1_npy, 'f1_csv': f1_csv,
                           'preds_npy': preds_npy, 'preds_csv': preds_csv}
    print(f'  Test .npy : Acc={acc_npy:.2%}  F1={f1_npy:.3f}')
    print(f'  Test CSV  : Acc={acc_csv:.2%}  F1={f1_csv:.3f}')
    print()

---
## 📊 Section 5 — Comparaison avec les baselines (spectres bruts / PCA)


In [ ]:
# ── Résultats de référence déjà obtenus (sweep RF/GB "mixte", fraction=1.0) ──
baseline_spectra_pca = {
    'RF'  : {'acc_npy': 0.9329, 'acc_csv': 0.9800},
    'XGB' : {'acc_npy': None, 'acc_csv': None},   # à compléter si tu as le détail par modèle
    'LGB' : {'acc_npy': None, 'acc_csv': None},
}
# NOTE : le sweep RF/GB précédent gardait seulement le MEILLEUR modèle/feature set
# par run — si tu veux la décomposition XGB seul / LGB seul sur spectres/PCA,
# il faudra les relire depuis les prints du run 'rfgb_mixed_csvfrac100'

# ── PatchTST classification complète (référence) ──────────────────────────
patchtst_full = {'acc_npy': 0.9503, 'acc_csv': 0.9850}

print('═'*70)
print('  COMPARAISON — Embeddings SSL vs Spectres bruts/PCA vs PatchTST complet')
print('═'*70)
print(f'{"Approche":40s} | {"Test .npy":>10} | {"Test CSV":>10}')
print('-'*70)
print(f'{"Meilleur RF/GB sur spectres/PCA (réf.)":40s} | {baseline_spectra_pca["RF"]["acc_npy"]:>9.2%} | {baseline_spectra_pca["RF"]["acc_csv"]:>9.2%}')
print(f'{"XGBoost sur embeddings SSL":40s} | {results["XGBoost"]["acc_npy"]:>9.2%} | {results["XGBoost"]["acc_csv"]:>9.2%}')
print(f'{"LightGBM sur embeddings SSL":40s} | {results["LightGBM"]["acc_npy"]:>9.2%} | {results["LightGBM"]["acc_csv"]:>9.2%}')
print(f'{"PatchTST classification complète":40s} | {patchtst_full["acc_npy"]:>9.2%} | {patchtst_full["acc_csv"]:>9.2%}')
print('═'*70)

In [ ]:
# ── Graphique comparatif ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels = ['RF/GB\nspectres/PCA', 'XGBoost\nembeddings SSL', 'LightGBM\nembeddings SSL', 'PatchTST\ncomplet']
colors = ['#ED7D31', '#70AD47', '#4472C4', '#7030A0']

npy_vals = [baseline_spectra_pca['RF']['acc_npy'], results['XGBoost']['acc_npy'],
            results['LightGBM']['acc_npy'], patchtst_full['acc_npy']]
csv_vals = [baseline_spectra_pca['RF']['acc_csv'], results['XGBoost']['acc_csv'],
            results['LightGBM']['acc_csv'], patchtst_full['acc_csv']]

bars0 = axes[0].bar(labels, [v*100 for v in npy_vals], color=colors, edgecolor='white')
axes[0].bar_label(bars0, fmt='%.1f%%', padding=3, fontweight='bold')
axes[0].set_title('Test .npy (officiel)', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_ylim(0, 105)
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].get_xticklabels(), rotation=20, ha='right')

bars1 = axes[1].bar(labels, [v*100 for v in csv_vals], color=colors, edgecolor='white')
axes[1].bar_label(bars1, fmt='%.1f%%', padding=3, fontweight='bold')
axes[1].set_title('Test CSV (réel)', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 105)
axes[1].grid(axis='y', alpha=0.3)
plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right')

plt.suptitle('Qualité des embeddings SSL — Validation demandée par Dr. Sarun',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(HOME, 'comparison_ssl_embeddings.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Conclusion automatique ──────────────────────────────────────────────
emb_better_npy = (results['XGBoost']['acc_npy'] >= baseline_spectra_pca['RF']['acc_npy'] or
                  results['LightGBM']['acc_npy'] >= baseline_spectra_pca['RF']['acc_npy'])
emb_better_csv = (results['XGBoost']['acc_csv'] >= baseline_spectra_pca['RF']['acc_csv'] or
                  results['LightGBM']['acc_csv'] >= baseline_spectra_pca['RF']['acc_csv'])

print('═'*65)
print('  CONCLUSION — Réponse à la question de Dr. Sarun')
print('═'*65)
if emb_better_npy and emb_better_csv:
    print('  ✓ Les embeddings SSL sont AU MOINS AUSSI BONS que les spectres bruts/PCA')
    print('    sur les DEUX domaines → le SSL capture bien les features clés.')
elif emb_better_npy or emb_better_csv:
    print('  ~ Les embeddings SSL sont meilleurs sur UN domaine seulement.')
else:
    print('  ⚠️  Les embeddings SSL sont MOINS bons que les spectres bruts/PCA')
    print('     sur les deux domaines — le SSL seul (sans fine-tuning supervisé)')
    print('     ne capture pas encore une représentation optimale pour ces classes.')
print('═'*65)

In [ ]:
# ── Logger dans le dashboard dédié ──────────────────────────────────────
from torch.utils.tensorboard import SummaryWriter

HPARAMS_EMB = os.path.join(HOME, 'runs', 'hparams_ssl_embeddings_test')

def log_run(cfg, metrics, run_label):
    run_dir = os.path.join(HPARAMS_EMB, run_label)
    w = SummaryWriter(run_dir)
    hparams_clean = {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))}
    w.add_hparams(hparams_clean, metrics)
    w.close()
    print(f'✓ Run "{run_label}" loggé')

for model_name in ['XGBoost', 'LightGBM']:
    log_run(
        cfg={'model': model_name, 'embedding_dim': CFG['d_model'], 'pooling': 'mean'},
        metrics={
            'test_npy_acc': results[model_name]['acc_npy'],
            'test_csv_acc': results[model_name]['acc_csv'],
        },
        run_label=f'ssl_embeddings_{model_name.lower()}',
    )
print(f'\nDashboard : tensorboard --logdir {HPARAMS_EMB} --port 6012')